In [21]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [22]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.sample(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
162,871201,M,19.590,18.15,130.70,1214.0,0.11200,0.16660,0.25080,0.12860,...,26.39,174.90,2232.0,0.1438,0.3846,0.68100,0.22470,0.3643,0.09223,NaN
438,909231,B,13.850,19.60,88.68,592.6,0.08684,0.06330,0.01342,0.02293,...,28.01,100.90,749.1,0.1118,0.1141,0.04753,0.05890,0.2513,0.06911,NaN
110,864033,B,9.777,16.99,62.50,290.2,0.10370,0.08404,0.04334,0.01778,...,21.47,71.68,367.0,0.1467,0.1765,0.13000,0.05334,0.2533,0.08468,NaN
481,91227,B,13.900,19.24,88.73,602.9,0.07991,0.05326,0.02995,0.02070,...,26.42,104.40,830.5,0.1064,0.1415,0.16730,0.08150,0.2356,0.07603,NaN
91,861799,M,15.370,22.76,100.20,728.2,0.09200,0.10360,0.11220,0.07483,...,25.84,107.50,830.9,0.1257,0.1997,0.28460,0.14760,0.2556,0.06828,NaN


In [23]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [24]:
df.sample(5)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
33,M,19.27,26.47,127.90,1162.0,0.09401,0.17190,0.16570,0.07593,0.1853,...,24.15,30.90,161.40,1813.0,0.1509,0.6590,0.6091,0.17850,0.3672,0.11230
256,M,19.55,28.77,133.60,1207.0,0.09260,0.20630,0.17840,0.11440,0.1893,...,25.05,36.27,178.60,1926.0,0.1281,0.5329,0.4251,0.19410,0.2818,0.10050
405,B,10.94,18.59,70.39,370.0,0.10040,0.07460,0.04944,0.02932,0.1486,...,12.40,25.58,82.76,472.4,0.1363,0.1644,0.1412,0.07887,0.2251,0.07732
373,M,20.64,17.35,134.80,1335.0,0.09446,0.10760,0.15270,0.08941,0.1571,...,25.37,23.17,166.80,1946.0,0.1562,0.3055,0.4159,0.21120,0.2689,0.07055
489,M,16.69,20.20,107.10,857.6,0.07497,0.07112,0.03649,0.02307,0.1846,...,19.18,26.56,127.30,1084.0,0.1009,0.2920,0.2477,0.08737,0.4677,0.07623


#### Train Test Split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

#### Scaling

In [26]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### Label Encoding

In [27]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

#### Numpy Arrays to PyTorch Tensors


In [28]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_train.astype(np.float32))

#### Defining the model

In [29]:
class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)

    return out

#### Important Parameters

In [30]:
learning_rate = 0.1
epochs = 25

In [31]:
# Define loss function
loss_function = nn.BCELoss()

### Training Pipeline

In [32]:
# Create Model
model = MySimpleNN(X_train_tensor.shape[1])

# Define Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Define Loop
for epoch in range(epochs):
  # Forward Pass
  y_pred = model(X_train_tensor)

  # Loss Calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1))

  # Clear Gradients
  optimizer.zero_grad()

  # Backward Pass
  loss.backward()

  # Parameters Update
  optimizer.step()

  # Print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.5062116980552673
Epoch: 2, Loss: 0.42299166321754456
Epoch: 3, Loss: 0.37125420570373535
Epoch: 4, Loss: 0.3356610834598541
Epoch: 5, Loss: 0.3094353973865509
Epoch: 6, Loss: 0.28914323449134827
Epoch: 7, Loss: 0.2728605270385742
Epoch: 8, Loss: 0.25942495465278625
Epoch: 9, Loss: 0.24809208512306213
Epoch: 10, Loss: 0.23836220800876617
Epoch: 11, Loss: 0.22988678514957428
Epoch: 12, Loss: 0.22241507470607758
Epoch: 13, Loss: 0.2157614678144455
Epoch: 14, Loss: 0.2097855806350708
Epoch: 15, Loss: 0.2043788880109787
Epoch: 16, Loss: 0.1994561403989792
Epoch: 17, Loss: 0.19494910538196564
Epoch: 18, Loss: 0.19080261886119843
Epoch: 19, Loss: 0.18697135150432587
Epoch: 20, Loss: 0.18341779708862305
Epoch: 21, Loss: 0.18011042475700378
Epoch: 22, Loss: 0.1770225465297699
Epoch: 23, Loss: 0.17413142323493958
Epoch: 24, Loss: 0.17141753435134888
Epoch: 25, Loss: 0.1688639372587204


### Evaluation

In [33]:
# Model Evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.5392134189605713
